In [ ]:
# Execute this for split-screen desktop.

import os
from IPython.display import display, HTML
from sidecar import Sidecar

try:
    JUPYTERHUB_USER = os.environ['JUPYTERHUB_USER']
except KeyError:
    JUPYTERHUB_USER = None
url_prefix = f"/user/{JUPYTERHUB_USER}" if JUPYTERHUB_USER is not None else ''
remote_desktop_url = f"{url_prefix}/desktop"
sc = Sidecar(title='Desktop')
with sc:
    # The inserted custom HTML and CSS snippets are to make the tab resizable
    display(HTML(f"""
        <style>
        body.p-mod-override-cursor div.iframe-widget {{
            position: relative;
            pointer-events: none;

        }}

        body.p-mod-override-cursor div.iframe-widget:before {{
            content: '';
            position: absolute;
            top: 0;
            left: 0;
            right: 0;
            bottom: 0;
            background: transparent;
        }}
        </style>
        <div class="iframe-widget" style="width: calc(100% + 10px);height:100%;">
            <iframe src="{remote_desktop_url}" width="100%" height="100%"></iframe>
        </div>
    """))

In [ ]:
### Task 2 & Task 3

In Task 2, a turtle model was created in RViz using multiple primitive marker shapes. The model consists of a lower body, an upper level, and four wheels. Each part was implemented using basic shapes such as cubes and cylinders. The markers were carefully positioned and scaled to resemble a small vehicle-like turtle structure.

In Task 3, dynamic behavior was added by using TF transformations. Two markers (represented as spheres) were attached to the coordinate frames of `turtle1` and `turtle2`. By setting the `frame_id` of each marker to the respective turtle frame and enabling `frame_locked`, the markers automatically followed the movement of the turtles.

As the turtles move in the simulation, their corresponding markers move synchronously in RViz. This demonstrates how TF enables real-time transformation between coordinate frames and ensures consistent visualization of moving objects.

The combination of marker-based modeling (Task 2) and TF-based motion tracking (Task 3) illustrates how ROS2 can be used to create and visualize dynamic robotic systems in RViz.

# Assignment 2 - Coordinates & Transforms

## Build your Turtlebot

The task of this assignment is to build the TortugaBot through Visualization Markers. Lets first set up the environment.

Open Rviz. ([references](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/RViz/RViz-User-Guide/RViz-User-Guide.html))
```bash
rviz2
```
* Set Global options > Fixed frame to `/world`. 
* Then add TF.

Start the turtlesim demo from [the TF2 tutorials](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/Tf2/Introduction-To-Tf2.html) in one terminal here:
```bash
ros2 launch turtle_tf2_py turtle_tf2_demo.launch.py
```
and in another terminal run
```bash
ros2 run turtlesim turtle_teleop_key
```
Move turtle with arrow keys in the terminal.

Execute the below python code blocks to publish arrow on the turtle1 frame

https://wiki.ros.org/rviz/DisplayTypes/Marker


## Example: How to publish markers

In [ ]:
import rclpy
from rclpy.node import Node
from visualization_msgs.msg import Marker

In Rviz `Add` > `By topic` > `/balloon` > Marker

There should now be a red baloon above `turtle1`. To publish continually, spin the node.

Clean up afterwards

## Example: How to use rotation

In [ ]:
import sys
!{sys.executable} -m pip install transforms3d

In [ ]:
from tf_transformations import euler_matrix, quaternion_from_matrix, rotation_from_matrix, euler_from_matrix

In [ ]:
# Standard rotation
r = euler_matrix(0,0,1.57)

In [ ]:
# All rotations are represented as matrices underneath.
print(r)

In [ ]:
# Quaternion
quaternion_from_matrix(r)

In [ ]:
# Angle Rotations
angle, _, _ = rotation_from_matrix(r)
angle

In [ ]:
# Euler Angle
euler_from_matrix(r)

In [ ]:
import numpy as np

In [ ]:
r = euler_matrix(np.pi,0,0)

In [ ]:
q = quaternion_from_matrix(r)

In [ ]:
ax, ay, az, w = q

In [ ]:
marker = Marker()
marker.header.frame_id = '/world'
marker.type = marker.ARROW
marker.id = 99
marker.action = marker.ADD
marker.scale.x = 0.5
marker.scale.y = 0.1
marker.scale.z = 0.1
marker.color.r = 0.0
marker.color.g = 1.0
marker.color.b = 0.0
marker.color.a = 1.0
marker.pose.orientation.x = ax
marker.pose.orientation.y = ay
marker.pose.orientation.z = az
marker.pose.orientation.w = w

In [ ]:
import rclpy
from rclpy.node import Node
from visualization_msgs.msg import Marker

rclpy.init()
marker_pub = Node("example_rotation_publisher")
marker_pub.publisher_ = marker_pub.create_publisher(Marker, 'example_rotations', 10)
marker.header.stamp = marker_pub.get_clock().now().to_msg()
marker_pub.publisher_.publish(marker)

In [ ]:
import time
from copy import copy

num_arrows = 20
bow_split = np.pi * 2 / num_arrows

for i in range(0, num_arrows):
    r = euler_matrix(0, bow_split * i, 0)
    ax, ay, az, w = quaternion_from_matrix(r)
    marker.pose.orientation.x = ax
    marker.pose.orientation.y = ay
    marker.pose.orientation.z = az
    marker.pose.orientation.w = w
    marker.id = 100 + i
    marker_pub.publisher_.publish(marker)
    time.sleep(0.1)

Task 1 – Description

In this task, a ROS2 node was implemented to visualize markers in RViz. For this purpose, a publisher was created that publishes messages of type visualization_msgs/Marker on the topic /balloon.

The marker was defined as a sphere (SPHERE) and positioned in the coordinate frame of the turtle (turtle1 and turtle2). By using frame_locked = True, the marker automatically follows the movement of the respective turtle.

A timer ensures that the marker is updated and published regularly (approximately every 0.5 seconds). In RViz, the markers are displayed via the /balloon topic.

Additionally, two markers were created:

a red marker for turtle1
a blue marker for turtle2

Due to the TF transformations, the markers move synchronously with the turtles in the simulation.

For illustration, a screenshot was created showing the visualization of the markers in RViz as well as the movement of the turtles.

# Tasks

In [ ]:
## Task 1: Coordinate Frames & TF (2 Pts)

1. In your code, you set marker.header.frame_id = 'turtle1'. If the turtle moves to coordinates (x=5,y=5) in the world frame, and your marker.pose.position.z is set to 1.0, where will the balloon appear in RViz? Explain the benefit of "parenting" the marker to the turtle frame instead of the world frame.

2. If you change the Fixed Frame in RViz to turtle1 while the turtle is spinning/moving, how will the grid and the balloon appear to behave compared to when the Fixed Frame is set to world?

3. Why does RViz show a "Status: Error" for a marker if the turtle_tf2_demo node is not running, even if your notebook is successfully publishing the /balloon topic?

In [ ]:
## Task 2 (6 Pts)

Below you can see one example of a turtle. It is approx. 40cm long. Use multiple primitive shapes to build the shape of the turtle. Your code should be running without executing the examples above.

The turtle has at least 2 floors and 4 wheels.

![turtle.png](attachment:02cbeaf5-f97c-44b1-b883-7f58f2fcc7cb.png)

In [ ]:
"""
import os
os.environ['ROS_DOMAIN_ID'] = '2'
print(os.environ['ROS_DOMAIN_ID']) # This should now print 2
"""

In [ ]:
# Your code
import rclpy
from rclpy.node import Node
from visualization_msgs.msg import Marker

class TurtleBuilder(Node):
    def __init__(self):
        super().__init__('turtle_builder')

        self.publisher_ = self.create_publisher(Marker, 'turtle_shape', 10)
        self.timer = self.create_timer(0.5, self.publish_turtle)

    def publish_turtle(self):
        now = self.get_clock().now().to_msg()

        # 🟩 Body (untere Ebene)
        body = Marker()
        body.header.frame_id = 'world'
        body.header.stamp = now
        body.id = 0
        body.type = Marker.CUBE
        body.action = Marker.ADD
        body.scale.x = 1.0
        body.scale.y = 0.5
        body.scale.z = 0.3
        body.color.r = 0.0
        body.color.g = 1.0
        body.color.b = 0.0
        body.color.a = 1.0
        body.pose.position.z = 0.15

        self.publisher_.publish(body)

        # 🟩 Obergeschoss
        top = Marker()
        top.header.frame_id = 'world'
        top.header.stamp = now
        top.id = 1
        top.type = Marker.CUBE
        top.action = Marker.ADD
        top.scale.x = 0.6
        top.scale.y = 0.4
        top.scale.z = 0.3
        top.color.r = 0.0
        top.color.g = 0.5
        top.color.b = 0.0
        top.color.a = 1.0
        top.pose.position.z = 0.5

        self.publisher_.publish(top)

        # ⚫ 4 Räder
        wheel_positions = [
            (-0.4, -0.3),
            (-0.4,  0.3),
            ( 0.4, -0.3),
            ( 0.4,  0.3),
        ]

        for i, (x, y) in enumerate(wheel_positions):
            wheel = Marker()
            wheel.header.frame_id = 'world'
            wheel.header.stamp = now
            wheel.id = 10 + i
            wheel.type = Marker.CYLINDER
            wheel.action = Marker.ADD
            wheel.scale.x = 0.2
            wheel.scale.y = 0.2
            wheel.scale.z = 0.1
            wheel.color.r = 0.0
            wheel.color.g = 0.0
            wheel.color.b = 0.0
            wheel.color.a = 1.0
            wheel.pose.position.x = x
            wheel.pose.position.y = y
            wheel.pose.position.z = 0.05

            self.publisher_.publish(wheel)


# ▶️ Start
rclpy.init()
node = TurtleBuilder()
rclpy.spin(node)

In [ ]:
class Turtle(Node):
    
    def __init__(self, name, id_seed):
        # Initialize the turtle node including its publisher
        
        pass
    
    def publish_turtle(self):
        # Publish the turtle as marker
        pass

In [ ]:
turtle1 = Turtle("turtle1", 100)
turtle1.publish_turtle()

## Task 3 (4 Pts)

Make a marker for turtle1 and turtle2 and let them follow the position of their respective turtles continuously.

In [ ]:
import rclpy
from rclpy.node import Node
from visualization_msgs.msg import Marker

class Balloon(Node):
    def __init__(self):
        super().__init__('balloon_marker')
        self.publisher_ = self.create_publisher(Marker, 'balloon', 10)
        self.timer = self.create_timer(0.5, self.publish_marker)

        self.marker = Marker()
        self.marker.header.frame_id = 'turtle1'
        self.marker.frame_locked = True
        self.marker.type = Marker.SPHERE
        self.marker.id = 0
        self.marker.action = Marker.ADD
        self.marker.scale.x = 0.4
        self.marker.scale.y = 0.4
        self.marker.scale.z = 0.4
        self.marker.color.r = 1.0
        self.marker.color.a = 1.0
        self.marker.pose.position.z = 0.5

        self.marker2 = Marker()
        self.marker2.header.frame_id = 'turtle2'
        self.marker2.frame_locked = True
        self.marker2.type = Marker.SPHERE
        self.marker2.id = 1
        self.marker2.action = Marker.ADD
        self.marker2.scale.x = 0.4
        self.marker2.scale.y = 0.4
        self.marker2.scale.z = 0.4
        self.marker2.color.b = 1.0
        self.marker2.color.a = 1.0
        self.marker2.pose.position.z = 0.5

    def publish_marker(self):
        now = self.get_clock().now().to_msg()
        self.marker.header.stamp = now
        self.marker2.header.stamp = now
        self.publisher_.publish(self.marker)
        self.publisher_.publish(self.marker2)

rclpy.init()
balloon = Balloon()
rclpy.spin(balloon)